<a href="https://colab.research.google.com/github/EthanJF-Physics/MAT422/blob/main/MAT422HW3EthanF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# MAT422 HW1.4
import matplotlib.pyplot as plt
import numpy as np
# Set the matrix size
matrix_size = 3
# Function to generate a random square matrix
def random_matrix(size: int = 3):
    return np.random.randint(0, 10, [size, size])
# Generate random matrices
A = random_matrix(matrix_size)
B = random_matrix(matrix_size)
print("Random 3x3 matrix A:")
print(A)
print("\nRandom 3x3 matrix B:")
print(B)
# Generate a random vector
rng = np.random.default_rng()
b = rng.random(3)
print("\nRandom 3x1 vector")
print(b)

Random 3x3 matrix A:
[[8 7 0]
 [4 5 7]
 [0 5 9]]

Random 3x3 matrix B:
[[0 0 4]
 [4 5 8]
 [3 1 1]]

Random 3x1 vector
[0.15351388 0.2877675  0.66371564]


1.4.1: Singular Value Decomp
Let $A$ be an $m \times n$ matrix. The square matrix $A^T A$ is symmetric and can be orthogonally diagonalized. Let $v_1, \dots, v_n$ be an orthonormal basis for $\mathbb{R}^n$ consisting of eigenvectors of $A^T A$, and let $\lambda_1, \dots, \lambda_n$ be the associated eigenvalues arranged in decreasing order:
$$ \lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_n \ge 0 $$

The singular values of $A$ (denoted by $\sigma_1, \dots, \sigma_n$) are the square roots of these eigenvalues and represent the lengths of the mapped vectors $A v_i$:
$$ \sigma_i = \sqrt{\lambda_i} = \|A v_i\| \quad \text{for } 1 \le i \le n $$

**Theorem 1.4.1.** If an $m \times n$ matrix $A$ has $r$ non-zero singular values ($\sigma_1, \dots, \sigma_r > 0$ with $\sigma_{r+1} = \dots = \sigma_n = 0$), then the dimension of $\text{col}(A) = r$ (the matrix rank).

The decomposition of $A$ involves an $m \times n$ diagonal-block matrix $\Sigma$ of the form:
$$ \Sigma = \begin{bmatrix} D & 0 \\ 0 & 0 \end{bmatrix} $$
where $D$ is an $r \times r$ diagonal matrix containing the non-zero singular values $\text{diag}(\sigma_1, \dots, \sigma_r)$.

**Theorem 1.4.2.** Let $A$ be an $m \times n$ matrix with $\text{rank}(A) = r$. Then there exists an $m \times n$ matrix $\Sigma$, an $m \times m$ orthogonal matrix $U$, and an $n \times n$ orthogonal matrix $V$ such that:
$$ A = U \Sigma V^T $$

#### Algorithmic Matrix Construction:
1. **Right Singular Vectors ($V$):** Columns are the sorted, normalized eigenvectors of $A^T A$.
   $$ V = \begin{bmatrix} v_1 & v_2 & \dots & v_n \end{bmatrix} $$
2. **Left Singular Vectors ($U$):** For $1 \le i \le r$, compute columns by normalizing $A v_i$. Then, extend to an orthonormal basis for $\mathbb{R}^m$:
   $$ u_i = \frac{1}{\sigma_i} A v_i \quad \implies \quad U = \begin{bmatrix} u_1 & u_2 & \dots & u_m \end{bmatrix} $$
3. **Singular Matrix ($\Sigma$):** An $m \times n$ matrix matching the dimensions of $A$, with $\sigma_i$ along the main diagonal and zeros elsewhere.

In [2]:
def svd(A):
    # Step 1: Compute A^T A
    ATA = A.T @ A
    # Step 2: Find eigenvalues and eigenvectors of A^T A
    eigenvalues_V, V = np.linalg.eigh(ATA)
    # Sort eigenvalues/eigenvectors from largest to smallest
    idx_V = np.argsort(eigenvalues_V)[::-1]
    eigenvalues_V = eigenvalues_V[idx_V]
    V = V[:, idx_V]
    # Step 3: Singular values = sqrt(eigenvalues)
    singular_values = np.sqrt(np.clip(eigenvalues_V, 0, None))
    # Step 4: Compute U
    U_cols = []
    for i in range(len(singular_values)):
        if singular_values[i] > 1e-9:
            u_i = (A @ V[:, i]) / singular_values[i]
            U_cols.append(u_i)

    U = np.column_stack(U_cols)

    return U, singular_values, V.T

# Use SVD function
U, s, VT = svd(A)
print("U:")
print(U)
print("\nSingular Values:")
print(s)
print("\nV Transpose:")
print(VT)
# Construct Sigma
Sigma = np.diag(s)
print("\nSigma:")
print(Sigma)
# Test Theorem 1.4.2: Does U @ Sigma @ V^T reconstruct A?
A_reconstructed = U @ Sigma @ VT
print("\nReconstructed A:")
print(A_reconstructed)
print("\nOriginal A:")
print(A)
print("\nIs your reconstruction successful?",
      np.allclose(A, A_reconstructed))


U:
[[-0.52639167  0.79836786  0.29243902]
 [-0.6173837  -0.12242189 -0.77707802]
 [-0.58459318 -0.58959448  0.55734116]]

Singular Values:
[15.18062775  8.76805079  1.29221762]

V Transpose:
[[-0.44007852 -0.63861826 -0.63126668]
 [ 0.67258453  0.23135053 -0.70292744]
 [-0.59494618  0.73392347 -0.32771236]]

Sigma:
[[15.18062775  0.          0.        ]
 [ 0.          8.76805079  0.        ]
 [ 0.          0.          1.29221762]]

Reconstructed A:
[[ 8.00000000e+00  7.00000000e+00 -1.45154593e-16]
 [ 4.00000000e+00  5.00000000e+00  7.00000000e+00]
 [-1.10532139e-15  5.00000000e+00  9.00000000e+00]]

Original A:
[[8 7 0]
 [4 5 7]
 [0 5 9]]

Is your reconstruction successful? True


### 1.4.2: Low-Rank Matrix Approximations
Matrix norms allow us to quantify and discuss the distance between two matrices.

**Definition 1.4.3 (Induced Norm).** The $2$-norm of an $n \times m$ matrix $A \in \mathbb{R}^{n \times m}$ is defined as:
$$ \|A\|_2 = \max_{0 \neq x \in \mathbb{R}^m} \frac{\|Ax\|}{\|x\|} = \max_{x \neq 0, \|x\|=1} \|Ax\| = \max_{x \neq 0, \|x\|=1} x^T A^T A x $$

Let $A \in \mathbb{R}^{n \times m}$ be a matrix with a Singular Value Decomposition (SVD):
$$ A = \sum_{j=1}^r \sigma_j u_j v_j^T $$

For any rank $k < r$, we can truncate this sum at the $k$-th term to form a low-rank approximation matrix $A_k$:
$$ A_k = \sum_{j=1}^k \sigma_j u_j v_j^T $$

The rank of $A_k$ is exactly $k$. This is true by construction because:
1. **Orthonormal Vectors:** The left singular vectors $\{u_j : j = 1, \dots, k\}$ are orthonormal.
2. **Column Space Basis:** Since the singular values $\sigma_j > 0$ and the right singular vectors $\{v_j : j = 1, \dots, k\}$ are orthonormal, the set $\{u_j : j = 1, \dots, k\}$ forms a basis that spans the column space of $A_k$.

**Lemma 1.4.4 (Matrix Norms and Singular Values).** Let $A \in \mathbb{R}^{n \times m}$ be a matrix with an SVD, where the singular values are ordered such that $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$. If $A_k$ is the $k$-th truncation defined above, then the squared $2$-norm error of the approximation satisfies:
$$ \|A - A_k\|_2^2 = \sigma_{k+1}^2 $$

**Theorem 1.4.5 (Eckart–Young–Mirsky Theorem).** Let $A \in \mathbb{R}^{n \times m}$ be a matrix with an SVD, and let $A_k$ be its $k$-th truncation where $k < r$. For any matrix $B \in \mathbb{R}^{n \times m}$ with $\text{rank}(B) \le k$:
$$ \|A - A_k\|_2 \le \|A - B\|_2 $$

In [3]:
# Set truncation rank k, and for a 3x3 full-rank matrix, k must satisfy k < 3
k = 2
print(f"Setting target approximation rank k = {k}")

# Compute A_k using the Outer Product Expansion:  A_k = sum_{j=1}^k sigma_j u_j v_j^T
A_k = np.zeros_like(A, dtype=float)
for j in range(k):
    A_k += s[j] * np.outer(U[:, j], VT[j, :])
print(f"\nRank-{k} Approximation Matrix A_k:")
print(A_k)
print(f"Calculated Rank of A_k:")
print(np.linalg.matrix_rank(A_k))

# Verify Lemma 1.4.4:  ||A - A_k||_2^2 = sigma_(k+1)^2
error_matrix = A - A_k
induced_2_norm_error = np.linalg.norm(error_matrix, ord=2)
squared_error = induced_2_norm_error ** 2
# Python uses zero-based indexing:  s[k] = sigma_(k+1)
expected_squared_error = s[k] ** 2
print("\n--- Verifying Lemma 1.4.4 ---")
print(f"Squared 2-norm error ||A - A_k||_2^2: "
      f"{squared_error:.6f}")
print(f"Expected error sigma_(k+1)^2:          "
      f"{expected_squared_error:.6f}")
print(
    "Lemma 1.4.4 verified?",
    np.isclose(squared_error, expected_squared_error)
)

# Create an arbitrary rank-k comparison matrix B *This matrix will be used to test the Eckart-Young-Mirsky Theorem)
np.random.seed(42)
X = np.random.randn(A.shape[0], k)
Y = np.random.randn(k, A.shape[1])
B = X @ Y
print("\nRank of comparison matrix B:")
print(np.linalg.matrix_rank(B))

# Compare the approximation errors, Theorem 1.4.5 says: ||A - A_k||_2 <= ||A - B||_2 for every matrix B with rank(B) <= k.
norm_A_minus_Ak = np.linalg.norm(A - A_k, ord=2)
norm_A_minus_B = np.linalg.norm(A - B, ord=2)
print("\n--- Verifying Eckart-Young-Mirsky Theorem ---")
print(f"Best approximation error ||A - A_k||_2: "
      f"{norm_A_minus_Ak:.6f}")
print(f"Comparison matrix error ||A - B||_2:     "
      f"{norm_A_minus_B:.6f}")
print(
    "Theorem 1.4.5 holds?",
    norm_A_minus_Ak <= norm_A_minus_B + 1e-9
)

Setting target approximation rank k = 2

Rank-2 Approximation Matrix A_k:
[[8.2248271  6.7226541  0.12384082]
 [3.40258247 5.73697212 6.67092635]
 [0.42848385 4.47142386 9.23602043]]
Calculated Rank of A_k:
2

--- Verifying Lemma 1.4.4 ---
Squared 2-norm error ||A - A_k||_2^2: 1.669826
Expected error sigma_(k+1)^2:          1.669826
Lemma 1.4.4 verified? True

Rank of comparison matrix B:
2

--- Verifying Eckart-Young-Mirsky Theorem ---
Best approximation error ||A - A_k||_2: 1.292218
Comparison matrix error ||A - B||_2:     15.093256
Theorem 1.4.5 holds? True


### 1.4.3.2 Principal Component Analysis
The goal of principal component analysis (PCA) is to project a high-dimensional dataset onto a lower-dimensional subspace while retaining as much variance as possible.

Let the columns of the $p \times N$ data matrix $X$ represent $N$ observations in a $p$-dimensional space:
$$ X = [X_1, X_2, \cdots, X_N] $$

We assume that the dataset is already in mean-deviation form (centered around zero). The optimization objective of PCA is to find a set of $k$ ($k \le p$) orthonormal vectors $\{v_1, \dots, v_k\}$ that maximize the total variance of the projected data points:
$$ \max_{v_j} \frac{1}{N} \sum_{i=1}^N \sum_{j=1}^k (X_i \cdot v_j)^2 $$

The term $(X_i \cdot v_j)$ represents the scalar projection length of the observation $X_i$ onto the direction vector $v_j$. By rewriting this projection using matrix operations, we observe that for each direction $j$:
$$ v_j^T X X^T v_j = (X^T v_j)^T (X^T v_j) = \sum_{i=1}^N (X_i \cdot v_j)^2 $$

where $X X^T$ forms a $p \times p$ matrix proportional to the sample covariance matrix.

**Definition 1.4.6 (Variance Maximization Rephrasing).** For each component $j \le k$, the directional variance-maximization subproblem can be formulated as a constrained optimization problem over the unit sphere:
$$ \operatorname{argmax}_{\|v_j\|=1} v_j^T X X^T v_j $$

Let the matrix $X X^T$ possess the following spectral (eigenvalue) decomposition:
$$ X X^T = V \operatorname{diag}(\lambda_1, \dots, \lambda_p) V^T \quad \text{or} \quad V^T X X^T V = \operatorname{diag}(\lambda_1, \dots, \lambda_p) $$

By application of the Spectral Theorem, the optimal choices for the projection axes $v_1, \dots, v_k$ are the first $k$ eigenvectors of $X X^T$ corresponding to its $k$ largest eigenvalues ($\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_p$). These vectors form the first $k$ columns of $V = [v_1, \dots, v_p]$ and are defined as the **principal components** of the data.

### 1.4.3.3 Change of Variables and Total Variance
The orthogonal $p \times p$ matrix $V$ defines a linear transformation that changes our coordinate framework via $x = V y$:
$$ \begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_p \end{pmatrix} = \begin{pmatrix} v_1 & v_2 & \cdots & v_p \end{pmatrix} \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_p \end{pmatrix} $$

This transformation map decouples the features such that the transformed variables $y_1, \dots, y_p$ are completely uncorrelated and ordered by decreasing variance. This structural optimization holds because:
$$ x^T X X^T x = y^T V^T X X^T V y = y^T \operatorname{diag}(\lambda_1, \dots, \lambda_p) y = \sum_{i=1}^p \lambda_i y_i^2 $$

Since $V$ is orthogonal, its inverse matches its transpose ($V^{-1} = V^T$). The mapping equation $y = V^T x$ yields each coordinate $y_i$ explicitly:
$$ y_i = v_i^T x = v_{1i}x_1 + v_{2i}x_2 + \cdots + v_{pi}x_p $$

**Definition 1.4.7 (Loadings).** The coordinate $y_i$ is a linear combination of the original coordinate variables $x_1, \dots, x_p$. The scalar entries of the steering eigenvector $v_i$ acting as structural weights are defined as the **loadings**.

Let the true sample covariance matrix $S$ of our centered matrix $X$ be defined as:
$$ S = \frac{1}{N-1} X X^T $$

The diagonal entry $s_{jj}$ of the matrix $S = [S_{ij}]$ denotes the sample variance of the $j$-th original feature space component $x_j$.

**Definition 1.4.8 (Total Variance).** The total variance of the data equals the sum of the variances on the main diagonal of $S$. Using the matrix trace operator $\operatorname{tr}(\cdot)$, this equates to:
$$ \text{Total Variance} = \operatorname{tr}(S) $$

Using the cyclic properties of the trace operator ($\operatorname{tr}(V S V^T) = \operatorname{tr}(S)$), the total dataset variance can be computed directly from the spectrum of the transformation:
$$ \operatorname{tr}(S) = \frac{1}{N-1} \sum_{j=1}^p \lambda_j $$

When performing dimensional reduction, truncating the space to the first $k$ terms preserves a predictable portion of the information. The total fraction of explained variance captured by this $k$-term truncation is given by:
$$ \frac{\sum_{j=1}^k \lambda_j}{\sum_{j=1}^p \lambda_j} $$


In [4]:
# p = 4 features, N = 100 observations
p, N = 4, 100
rng = np.random.default_rng(seed=42)
raw_data = rng.normal(loc=5.0, scale=2.0, size=(p, N))
# Center the data (Dataset must be in mean-deviation form)
X = raw_data - np.mean(raw_data, axis=1, keepdims=True)
# Define target truncation rank
k = 2
print(f"Dataset Dimensions: {p} features, {N} observations.")
print(f"Target Subspace Rank k = {k}\n")
# Compute XX^T (proportional to the sample covariance matrix)
XX_T = X @ X.T
# Spectral (eigenvalue) decomposition of XX^T
eigenvalues, V = np.linalg.eigh(XX_T)
# Sort eigenvalues (lambda) and eigenvectors (V) in descending order
idx = np.argsort(eigenvalues)[::-1]
lambdas = eigenvalues[idx]
V = V[:, idx]
# Extract first k principal components (optimal projection axes)
principal_components = V[:, :k]
print("--- 1.4.3.2 Verification ---")
print("Top k largest eigenvalues (lambdas):", lambdas[:k])
print("First Principal Component (v_1):\n", principal_components[:, 0])

# The columns of V are the steering eigenvectors (loadings)
loadings = V  # Every column v_i contains the weights for coordinate y_i
# Transform the framework: y = V^T @ x (or Y = V^T @ X for all data points)
Y = V.T @ X
# Compute the sample covariance matrix S
S = (1 / (N - 1)) * (X @ X.T)
# Definition 1.4.8: Total Variance via trace of S
total_variance_trace = np.trace(S)
# Total Variance calculated from the spectrum (lambdas)
total_variance_spectrum = (1 / (N - 1)) * np.sum(lambdas)
# Fraction of explained variance captured by k-term truncation
explained_variance_ratio = np.sum(lambdas[:k]) / np.sum(lambdas)

print("\n--- 1.4.3.3 Verification ---")
print(f"Total Variance via tr(S):         {total_variance_trace:.6f}")
print(f"Total Variance via Spectrum sum:  {total_variance_spectrum:.6f}")
print(f"Are trace and spectrum equal?     {np.isclose(total_variance_trace, total_variance_spectrum)}")
print(f"Proportion of Variance Explained (k={k}): {explained_variance_ratio * 100:.2f}%")

# Verify that transformed variables Y are decoupled
cov_Y = (1 / (N - 1)) * (Y @ Y.T)
print("\nIs the transformed covariance matrix diagonal (uncorrelated)?")
print(np.allclose(cov_Y, np.diag(np.diag(cov_Y))))
# Verify that V is orthogonal
print("\nIs V orthogonal?")
print(np.allclose(V.T @ V, np.eye(p)))
# Verify that Y = V^T X can reconstruct X
X_reconstructed = V @ Y
print("\nCan V @ Y reconstruct X?")
print(np.allclose(X, X_reconstructed))
# Display covariance matrix of transformed variables
print("\nCovariance matrix of transformed variables:")
print(cov_Y)
# Display explained variance for each component
individual_explained_variance = lambdas / np.sum(lambdas)
print("\nExplained variance by each principal component:")
for i in range(p):
    print(
        f"PC{i+1}: "
        f"{individual_explained_variance[i] * 100:.2f}%"
    )

Dataset Dimensions: 4 features, 100 observations.
Target Subspace Rank k = 2

--- 1.4.3.2 Verification ---
Top k largest eigenvalues (lambdas): [450.82456559 409.57116203]
First Principal Component (v_1):
 [ 0.01526955 -0.53962535 -0.39761368  0.74193981]

--- 1.4.3.3 Verification ---
Total Variance via tr(S):         14.552931
Total Variance via Spectrum sum:  14.552931
Are trace and spectrum equal?     True
Proportion of Variance Explained (k=2): 59.72%

Is the transformed covariance matrix diagonal (uncorrelated)?
True

Is V orthogonal?
True

Can V @ Y reconstruct X?
True

Covariance matrix of transformed variables:
[[ 4.55378349e+00  9.48036992e-16 -9.98433354e-16 -5.26321924e-16]
 [ 9.48036992e-16  4.13708244e+00 -7.80824966e-16 -3.27666487e-16]
 [-9.98433354e-16 -7.80824966e-16  3.57033341e+00 -8.34090537e-16]
 [-5.26321924e-16 -3.27666487e-16 -8.34090537e-16  2.29173192e+00]]

Explained variance by each principal component:
PC1: 31.29%
PC2: 28.43%
PC3: 24.53%
PC4: 15.75%
